# DressCheck: Government Employee Attire Compliance Classifier

This notebook documents the machine learning workflow used by the DressCheck website. It trains an SVM classifier using HOG image features to classify government employee upper-garment images as either **Compliant** or **Non-Compliant**.

Presentation checklist covered here:
- Dataset loading and class counts
- Image preprocessing
- HOG feature extraction
- SVM training and evaluation
- Model artifact export for the website

## 1. Import Libraries

The project uses Pillow for image loading, scikit-image for HOG feature extraction, and scikit-learn for model training.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
from PIL import Image, UnidentifiedImageError
from skimage.feature import hog
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 2. Project Configuration

The dataset is expected to contain two folders:
- `Midterm Dataset/Compliant`
- `Midterm Dataset/Non-Compliant`

Each image is resized to `128 x 128` pixels before HOG feature extraction.

In [ ]:
DATASET_DIR = Path("Midterm Dataset")
MODEL_DIR = Path("models")
IMAGE_SIZE = (128, 128)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

CLASSES = {
    "Non-Compliant": 0,
    "Compliant": 1,
}

MODEL_DIR.mkdir(exist_ok=True)

## 3. Inspect Dataset

This verifies that both classes are available and counts the image files in each class folder.

In [ ]:
def image_paths(folder):
    return sorted(
        path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


for class_name in CLASSES:
    folder = DATASET_DIR / class_name
    paths = image_paths(folder)
    print(f"{class_name}: {len(paths)} images")

## 4. Feature Extraction

The model uses Histogram of Oriented Gradients (HOG), a feature descriptor that captures edge and shape patterns in an image. This is suitable for a traditional machine learning pipeline such as SVM.

In [ ]:
def extract_features(path):
    with Image.open(path) as img:
        arr = np.array(img.convert("RGB").resize(IMAGE_SIZE))

    return hog(
        arr,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        channel_axis=-1,
    )


def load_dataset():
    features = []
    labels = []
    skipped = []

    for class_name, label in CLASSES.items():
        folder = DATASET_DIR / class_name
        if not folder.exists():
            raise FileNotFoundError(f"Missing dataset folder: {folder}")

        for path in image_paths(folder):
            try:
                features.append(extract_features(path))
                labels.append(label)
            except (OSError, UnidentifiedImageError, ValueError) as exc:
                skipped.append((path, exc))

    if not features:
        raise RuntimeError("No valid training images were found.")

    return np.array(features), np.array(labels), skipped


X, y, skipped = load_dataset()
print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Skipped unreadable files: {len(skipped)}")

## 5. Train-Test Split and Scaling

The split is stratified so both classes keep similar proportions in the training and test sets. A `StandardScaler` is applied because SVM models are sensitive to feature scale.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training images: {len(X_train)}")
print(f"Testing images: {len(X_test)}")

## 6. Train SVM with Grid Search

Grid search compares multiple SVM settings and selects the best parameter combination using cross-validation.

In [ ]:
search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid={
        "kernel": ["rbf", "linear"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
        "class_weight": [None, "balanced"],
    },
    cv=5,
    n_jobs=1,
)

search.fit(X_train_scaled, y_train)
model = search.best_estimator_

print("Best parameters:")
print(search.best_params_)

## 7. Evaluate the Model

The test set is used to estimate how well the classifier performs on unseen images.

In [ ]:
y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Non-Compliant", "Compliant"]))

## 8. Train Final Model and Save Artifacts

After evaluation, the final model is trained using the full dataset. The website uses these two files:
- `models/svm_model.pkl`
- `models/scaler.pkl`

In [ ]:
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X)

final_model = SVC(probability=True, random_state=42, **search.best_params_)
final_model.fit(X_scaled, y)

with (MODEL_DIR / "svm_model.pkl").open("wb") as f:
    pickle.dump(final_model, f)

with (MODEL_DIR / "scaler.pkl").open("wb") as f:
    pickle.dump(final_scaler, f)

print("Saved models/svm_model.pkl")
print("Saved models/scaler.pkl")

## 9. Single Image Prediction Test

Use this cell to test one image manually. Change `sample_image` to any image path from the dataset or a new uploaded garment image.

In [ ]:
sample_image = next(image_paths(DATASET_DIR / "Compliant"))

features = extract_features(sample_image)
features_scaled = final_scaler.transform([features])
prediction = int(final_model.predict(features_scaled)[0])
probability = final_model.predict_proba(features_scaled)[0]
confidence = max(probability) * 100

label_name = "Compliant" if prediction == 1 else "Non-Compliant"
print(f"Image: {sample_image}")
print(f"Prediction: {label_name}")
print(f"Confidence: {confidence:.1f}%")

## 10. Notes for Presentation

- The classifier is a traditional machine learning model, not a deep learning model.
- The pipeline is: image upload -> resize to 128 x 128 -> HOG features -> scaler -> SVM prediction.
- The saved `.pkl` files are loaded by `api/predict.py` for the website demo.
- The website returns the predicted label and confidence score.